# Seasonal Agriculture Performance Analysis
**VOIS AICTE Batch1 2026-2027 — Major Project**

---

## Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading & Overview](#2-overview)
3. [Data Cleaning & Preparation](#3-cleaning)
4. [Exploratory Data Analysis (EDA)](#4-eda)
5. [Seasonal Performance Analysis](#5-seasonal)
6. [Environmental Conditions by Season](#6-environment)
7. [Resource Usage Analysis](#7-resources)
8. [Economic Performance Analysis](#8-economics)
9. [Crop-wise Seasonal Analysis](#9-crops)
10. [Correlation & Relationship Analysis](#10-correlation)
11. [Statistical Testing](#11-stats)
12. [Key Insights & Recommendations](#12-insights)

---
## Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. As a result, agricultural performance may differ from one season to another.

However, raw agricultural data does not clearly explain how agricultural performance changes across seasons or what patterns can be observed in different seasonal conditions.

**The problem:** Analyze the given agricultural dataset and investigate seasonal differences in agricultural performance by identifying meaningful patterns, trends, relationships and variations within the available data.

**Key Questions this notebook answers:**
- How does agricultural performance vary across seasons?
- What major seasonal patterns can be observed?
- Are there noticeable variations in resource usage across seasons?
- How do environmental conditions relate to agricultural performance?
- How do economic outcomes vary across seasons?
- Are there unusual or unexpected seasonal patterns?
- How could the findings support better seasonal agricultural planning?


---
## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# Core libraries
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Statistical testing
from scipy import stats

# Display settings
pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_theme(style='whitegrid', palette='Set2')

# Colour palette consistent across the notebook
SEASON_PALETTE = {'Kharif': '#2196F3', 'Rabi': '#4CAF50', 'Zaid': '#FF9800'}

print('All libraries imported successfully.')

---
## 2. Data Loading & Overview <a id='2-overview'></a>

In [ ]:
# Load dataset
df = pd.read_csv('seasonal_agriculture_performance_dataset.csv')

print(f'Dataset shape : {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Memory usage  : {df.memory_usage(deep=True).sum() / 1024:.1f} KB')
df.head()

In [ ]:
# Column names and data types
print('Column Info:')
df.info()

In [ ]:
# Summary statistics
df.describe().T

In [ ]:
# Distribution of key categorical columns
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col in zip(axes, ['Season', 'Crop', 'Irrigation_Method']):
    counts = df[col].value_counts()
    counts.plot(kind='bar', ax=ax, color=sns.color_palette('Set2', len(counts)))
    ax.set_title(f'Distribution of {col}', fontsize=13, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=30)
    for p in ax.patches:
        ax.annotate(str(int(p.get_height())),
                    (p.get_x() + p.get_width() / 2, p.get_height()),
                    ha='center', va='bottom', fontsize=9)

plt.suptitle('Dataset Overview — Categorical Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Data Cleaning & Preparation <a id='3-cleaning'></a>

In [ ]:
# Check missing values
missing = df.isnull().sum()
missing = missing[missing > 0]
missing_pct = (missing / len(df) * 100).round(2)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print('Columns with missing values:')
print(missing_df)

In [ ]:
# Visualise missing values
fig, ax = plt.subplots(figsize=(8, 3))
missing_df['Missing %'].plot(kind='barh', ax=ax, color='#E57373')
ax.set_xlabel('Missing %')
ax.set_title('Missing Value Percentage by Column', fontweight='bold')
for p in ax.patches:
    ax.annotate(f'{p.get_width():.2f}%',
                (p.get_width() + 0.05, p.get_y() + p.get_height() / 2),
                va='center', fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Strategy: fill numerical missing values with season-wise median
# (preserves seasonal patterns rather than global imputation)
cols_to_impute = ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']

df_clean = df.copy()
for col in cols_to_impute:
    season_median = df_clean.groupby('Season')[col].transform('median')
    df_clean[col] = df_clean[col].fillna(season_median)

print('Missing values after imputation:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])
print('\nNo missing values remain.' if df_clean.isnull().sum().sum() == 0 else '')

In [ ]:
# Check for duplicates
dup_count = df_clean.duplicated().sum()
print(f'Duplicate rows: {dup_count}')

# Check Farm_ID uniqueness
print(f'Unique Farm_IDs: {df_clean["Farm_ID"].nunique()} (total rows: {len(df_clean)})')

In [ ]:
# Detect outliers using IQR method for key numeric columns
numeric_cols = ['Yield_Tonnes_Ha', 'Profit_INR', 'Revenue_INR', 'Rainfall_mm',
                'Water_Used_m3', 'Fertilizer_kg_ha']

outlier_summary = []
for col in numeric_cols:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df_clean[col] < Q1 - 1.5 * IQR) | (df_clean[col] > Q3 + 1.5 * IQR)).sum()
    outlier_summary.append({'Column': col, 'Outlier Count': outliers,
                             'Outlier %': round(outliers / len(df_clean) * 100, 2)})

print('Outlier Summary (IQR method):')
print(pd.DataFrame(outlier_summary).to_string(index=False))
print('\nNote: Outliers are retained as they likely reflect real high-value / high-yield farms.')

In [ ]:
# Add derived features useful for analysis
df_clean['Profit_per_Ha'] = df_clean['Profit_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Revenue_per_Ha'] = df_clean['Revenue_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Cost_per_Ha'] = df_clean['Total_Cost_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Profitable'] = (df_clean['Profit_INR'] > 0).map({True: 'Profitable', False: 'Loss'})

print('Derived features added: Profit_per_Ha, Revenue_per_Ha, Cost_per_Ha, Profitable')
print(f'\nOverall profitability rate: {(df_clean["Profit_INR"] > 0).mean() * 100:.1f}%')

---
## 4. Exploratory Data Analysis (EDA) <a id='4-eda'></a>

In [ ]:
# Distribution of key numeric variables
key_vars = ['Yield_Tonnes_Ha', 'Profit_INR', 'Rainfall_mm',
            'Avg_Temperature_C', 'Fertilizer_kg_ha', 'Water_Used_m3']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for ax, var in zip(axes, key_vars):
    ax.hist(df_clean[var].dropna(), bins=40, color='#5C85D6', edgecolor='white', alpha=0.85)
    ax.axvline(df_clean[var].median(), color='red', linestyle='--', linewidth=1.5,
               label=f'Median: {df_clean[var].median():.1f}')
    ax.set_title(var, fontweight='bold')
    ax.legend(fontsize=8)

plt.suptitle('Distribution of Key Numeric Variables', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Yield distribution per season using KDE + histogram
fig, ax = plt.subplots(figsize=(12, 5))
for season, color in SEASON_PALETTE.items():
    data = df_clean[df_clean['Season'] == season]['Yield_Tonnes_Ha']
    ax.hist(data, bins=40, alpha=0.45, color=color, label=season, density=True)
    data.plot.kde(ax=ax, color=color, linewidth=2.5)

ax.set_xlabel('Yield (Tonnes/Ha)', fontsize=12)
ax.set_ylabel('Density')
ax.set_title('Yield Distribution by Season', fontsize=14, fontweight='bold')
ax.legend(title='Season', fontsize=11)
plt.tight_layout()
plt.show()

---
## 5. Seasonal Performance Analysis <a id='5-seasonal'></a>

In [ ]:
# Aggregate KPIs by season
season_kpis = df_clean.groupby('Season').agg(
    Farm_Count       = ('Farm_ID', 'count'),
    Avg_Yield        = ('Yield_Tonnes_Ha', 'mean'),
    Median_Yield     = ('Yield_Tonnes_Ha', 'median'),
    Avg_Production   = ('Production_Tonnes', 'mean'),
    Avg_Revenue      = ('Revenue_INR', 'mean'),
    Avg_Profit       = ('Profit_INR', 'mean'),
    Median_Profit    = ('Profit_INR', 'median'),
    Profitable_Rate  = ('Profitable', lambda x: (x == 'Profitable').mean() * 100)
).round(2)

print('Seasonal KPI Summary:')
season_kpis

In [ ]:
# Bar chart — Average Yield by Season
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [
    ('Avg_Yield',       'Average Yield (Tonnes/Ha)',    '#5C85D6'),
    ('Avg_Profit',      'Average Profit (INR)',         '#66BB6A'),
    ('Profitable_Rate', 'Profitability Rate (%)',        '#FF8A65'),
]

for ax, (metric, label, color) in zip(axes, metrics):
    bars = ax.bar(season_kpis.index, season_kpis[metric],
                  color=[SEASON_PALETTE[s] for s in season_kpis.index],
                  edgecolor='white', linewidth=0.8, alpha=0.9)
    ax.set_title(label, fontweight='bold', fontsize=12)
    ax.set_xlabel('Season')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                h + abs(h) * 0.01, f'{h:,.1f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Seasonal Performance — Key Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots — Yield and Profit by Season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Season', y='Yield_Tonnes_Ha', hue='Season',
            palette=SEASON_PALETTE, ax=axes[0], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[0].set_title('Yield Distribution by Season', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Yield (Tonnes/Ha)')

sns.boxplot(data=df_clean, x='Season', y='Profit_per_Ha', hue='Season',
            palette=SEASON_PALETTE, ax=axes[1], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[1].set_title('Profit per Hectare by Season', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Profit per Ha (INR)')
axes[1].axhline(0, color='red', linewidth=1.2, linestyle='--', label='Break-even')
axes[1].legend()

plt.suptitle('Yield & Profitability Distribution by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Profitability breakdown by season
profit_breakdown = df_clean.groupby(['Season', 'Profitable']).size().unstack(fill_value=0)
profit_pct = profit_breakdown.div(profit_breakdown.sum(axis=1), axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

profit_breakdown.plot(kind='bar', ax=axes[0], color=['#E57373', '#66BB6A'],
                      edgecolor='white', alpha=0.9)
axes[0].set_title('Farms: Profitable vs Loss-making by Season', fontweight='bold')
axes[0].set_xlabel('Season')
axes[0].tick_params(axis='x', rotation=0)

profit_pct.plot(kind='bar', ax=axes[1], color=['#E57373', '#66BB6A'],
                edgecolor='white', alpha=0.9)
axes[1].set_title('Profitability Rate (%) by Season', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=0)
axes[1].axhline(50, color='black', linewidth=1, linestyle='--')

plt.suptitle('Profitability Analysis by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(profit_pct.round(1))

---
## 6. Environmental Conditions by Season <a id='6-environment'></a>

In [ ]:
# Environmental KPIs by season
env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct',
            'Sunlight_Hours_Day', 'Soil_pH', 'Soil_Moisture_pct', 'Disease_Pest_Risk_pct']

env_summary = df_clean.groupby('Season')[env_cols].mean().round(2)
print('Environmental Conditions — Seasonal Averages:')
env_summary

In [ ]:
# Radar / heatmap of environmental conditions
env_norm = (env_summary - env_summary.min()) / (env_summary.max() - env_summary.min())

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Heatmap
sns.heatmap(env_norm.T, annot=env_summary.T, fmt='.1f',
            cmap='YlOrRd', ax=axes[0], linewidths=0.5,
            cbar_kws={'label': 'Normalised Score'})
axes[0].set_title('Environmental Conditions by Season\n(annotated with actual values)',
                   fontweight='bold')
axes[0].set_xlabel('Season')

# Grouped bar — Rainfall & Temperature
x = np.arange(len(env_summary.index))
w = 0.35
axes[1].bar(x - w/2, env_summary['Rainfall_mm'], width=w,
            label='Rainfall (mm)', color='#42A5F5', alpha=0.9)
ax2 = axes[1].twinx()
ax2.bar(x + w/2, env_summary['Avg_Temperature_C'], width=w,
        label='Avg Temp (°C)', color='#EF5350', alpha=0.9)
axes[1].set_xticks(x)
axes[1].set_xticklabels(env_summary.index)
axes[1].set_ylabel('Rainfall (mm)', color='#42A5F5')
ax2.set_ylabel('Avg Temperature (°C)', color='#EF5350')
axes[1].set_title('Rainfall vs Temperature by Season', fontweight='bold')
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.suptitle('Environmental Analysis by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots for key environmental variables
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
env_plot_vars = [
    ('Rainfall_mm',           'Rainfall (mm)'),
    ('Avg_Temperature_C',     'Avg Temperature (°C)'),
    ('Disease_Pest_Risk_pct', 'Disease & Pest Risk (%)'),
]
for ax, (var, label) in zip(axes, env_plot_vars):
    sns.violinplot(data=df_clean, x='Season', y=var, hue='Season',
                   palette=SEASON_PALETTE, ax=ax,
                   order=['Kharif', 'Rabi', 'Zaid'], inner='quartile', legend=False)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('')

plt.suptitle('Environmental Variable Distribution by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Resource Usage Analysis <a id='7-resources'></a>

In [ ]:
# Resource usage summary
resource_cols = ['Fertilizer_kg_ha', 'Pesticide_Litre_ha', 'Water_Used_m3',
                 'Water_Efficiency_t_per_1000m3', 'Nitrogen_kg_ha',
                 'Phosphorus_kg_ha', 'Potassium_kg_ha']

resource_summary = df_clean.groupby('Season')[resource_cols].mean().round(2)
print('Resource Usage — Seasonal Averages:')
resource_summary

In [ ]:
# Fertilizer and Pesticide usage by season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Season', y='Fertilizer_kg_ha', hue='Season',
            palette=SEASON_PALETTE, ax=axes[0], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[0].set_title('Fertilizer Usage by Season', fontweight='bold')
axes[0].set_ylabel('Fertilizer (kg/ha)')

sns.boxplot(data=df_clean, x='Season', y='Pesticide_Litre_ha', hue='Season',
            palette=SEASON_PALETTE, ax=axes[1], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[1].set_title('Pesticide Usage by Season', fontweight='bold')
axes[1].set_ylabel('Pesticide (Litres/ha)')

plt.suptitle('Input Resource Usage by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Irrigation method distribution by season
irr_season = df_clean.groupby(['Season', 'Irrigation_Method']).size().unstack(fill_value=0)
irr_pct = irr_season.div(irr_season.sum(axis=1), axis=0) * 100

irr_pct.plot(kind='bar', figsize=(10, 5), colormap='Set2',
             edgecolor='white', alpha=0.9)
plt.title('Irrigation Method Distribution by Season (%)', fontsize=13, fontweight='bold')
plt.xlabel('Season')
plt.ylabel('Percentage (%)')
plt.xticks(rotation=0)
plt.legend(title='Irrigation Method', bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

print(irr_pct.round(1))

In [ ]:
# Water efficiency by season and irrigation method
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Season', y='Water_Efficiency_t_per_1000m3', hue='Season',
            palette=SEASON_PALETTE, ax=axes[0], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[0].set_title('Water Efficiency by Season', fontweight='bold')
axes[0].set_ylabel('Water Efficiency (t / 1000 m³)')

sns.boxplot(data=df_clean, x='Irrigation_Method', y='Water_Efficiency_t_per_1000m3',
            hue='Irrigation_Method', palette='Set2', ax=axes[1], legend=False)
axes[1].set_title('Water Efficiency by Irrigation Method', fontweight='bold')
axes[1].set_ylabel('Water Efficiency (t / 1000 m³)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('Water Efficiency Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 8. Economic Performance Analysis <a id='8-economics'></a>

In [ ]:
# Revenue, Cost, Profit breakdown by season
econ_summary = df_clean.groupby('Season').agg(
    Total_Revenue = ('Revenue_INR', 'sum'),
    Total_Cost    = ('Total_Cost_INR', 'sum'),
    Total_Profit  = ('Profit_INR', 'sum'),
    Avg_Revenue   = ('Revenue_INR', 'mean'),
    Avg_Cost      = ('Total_Cost_INR', 'mean'),
    Avg_Profit    = ('Profit_INR', 'mean'),
).round(0)

print('Economic Summary by Season:')
econ_summary

In [ ]:
# Stacked bar — Avg Revenue vs Cost, with Profit overlay
fig, ax = plt.subplots(figsize=(10, 5))

seasons = econ_summary.index
x = np.arange(len(seasons))
w = 0.5

ax.bar(x, econ_summary['Avg_Revenue'], w, label='Avg Revenue', color='#42A5F5', alpha=0.85)
ax.bar(x, econ_summary['Avg_Cost'],    w, label='Avg Cost',    color='#EF5350', alpha=0.65)

# Profit points
ax.scatter(x, econ_summary['Avg_Profit'], color='#FFD600', s=120,
           zorder=5, label='Avg Profit', edgecolors='black', linewidth=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')

ax.set_xticks(x)
ax.set_xticklabels(seasons, fontsize=12)
ax.set_ylabel('INR')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'₹{v/1e5:.1f}L'))
ax.set_title('Average Revenue, Cost & Profit by Season', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Market price variation by season
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_clean, x='Season', y='Market_Price_INR_Tonne', hue='Season',
            palette=SEASON_PALETTE, ax=axes[0], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[0].set_title('Market Price by Season', fontweight='bold')
axes[0].set_ylabel('Market Price (INR/Tonne)')

# Revenue per hectare
sns.boxplot(data=df_clean, x='Season', y='Revenue_per_Ha', hue='Season',
            palette=SEASON_PALETTE, ax=axes[1], order=['Kharif', 'Rabi', 'Zaid'], legend=False)
axes[1].set_title('Revenue per Hectare by Season', fontweight='bold')
axes[1].set_ylabel('Revenue per Ha (INR)')

plt.suptitle('Economic Returns by Season', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9. Crop-wise Seasonal Analysis <a id='9-crops'></a>

In [ ]:
# Average yield per crop per season
crop_season_yield = df_clean.pivot_table(
    values='Yield_Tonnes_Ha', index='Crop', columns='Season', aggfunc='mean'
).round(2)

fig, ax = plt.subplots(figsize=(12, 5))
crop_season_yield.plot(kind='bar', ax=ax,
                       color=[SEASON_PALETTE[s] for s in crop_season_yield.columns],
                       edgecolor='white', alpha=0.9)
ax.set_title('Average Yield by Crop and Season', fontsize=13, fontweight='bold')
ax.set_xlabel('Crop')
ax.set_ylabel('Avg Yield (Tonnes/Ha)')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Season')
plt.tight_layout()
plt.show()

print('Average Yield (Tonnes/Ha) per Crop per Season:')
crop_season_yield

In [ ]:
# Heatmap — Avg Profit per Ha by Crop x Season
crop_profit_heat = df_clean.pivot_table(
    values='Profit_per_Ha', index='Crop', columns='Season', aggfunc='mean'
).round(0)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(crop_profit_heat, annot=True, fmt='.0f', cmap='RdYlGn',
            linewidths=0.5, ax=ax, center=0,
            cbar_kws={'label': 'Avg Profit per Ha (INR)'})
ax.set_title('Average Profit per Hectare — Crop × Season', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top performing crop each season
top_crops = df_clean.groupby(['Season', 'Crop'])['Profit_per_Ha'].mean().reset_index()
top_crops = top_crops.sort_values(['Season', 'Profit_per_Ha'], ascending=[True, False])

print('Top 3 Crops by Avg Profit/Ha for each Season:')
for season in ['Kharif', 'Rabi', 'Zaid']:
    top3 = top_crops[top_crops['Season'] == season].head(3)
    print(f'\n{season}:')
    print(top3[['Crop', 'Profit_per_Ha']].to_string(index=False))

In [ ]:
# State-wise performance heatmap (Avg Yield)
state_season = df_clean.pivot_table(
    values='Yield_Tonnes_Ha', index='State', columns='Season', aggfunc='mean'
).round(2)

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(state_season, annot=True, fmt='.2f', cmap='Blues',
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Avg Yield (Tonnes/Ha)'})
ax.set_title('Average Yield by State and Season', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 9b. Unusual & Unexpected Seasonal Patterns

Beyond expected seasonal trends, we flag **statistically extreme farms** (yield Z-score > 3 within their season) and investigate crops that perform unexpectedly well in the weakest season (Zaid). These unusual patterns reveal resilience factors and planning opportunities.


In [ ]:
# --- Unusual & Unexpected Seasonal Patterns ---
# Flag farms with yield > 3 standard deviations from their seasonal mean

season_stats = df_clean.groupby("Season")["Yield_Tonnes_Ha"].agg(["mean","std"]).rename(
    columns={"mean": "Season_Mean_Yield", "std": "Season_Std_Yield"}
)
df_anomaly = df_clean.merge(season_stats, on="Season")
df_anomaly["Yield_Z"] = (
    (df_anomaly["Yield_Tonnes_Ha"] - df_anomaly["Season_Mean_Yield"])
    / df_anomaly["Season_Std_Yield"]
)

extremes = df_anomaly[df_anomaly["Yield_Z"].abs() > 3]
print("Farms with unusually extreme yield (|Z| > 3):", len(extremes))
if len(extremes) > 0:
    print(extremes[["Farm_ID","Season","Crop","State","Yield_Tonnes_Ha","Yield_Z"]].to_string(index=False))

# Unexpected: Zaid farms that are profitable despite being the weakest season
zaid = df_clean[df_clean["Season"] == "Zaid"]
zaid_profitable = zaid[zaid["Profit_INR"] > 0]
zaid_pct = len(zaid_profitable) / len(zaid) * 100
print(f"\nUnexpected — Zaid profitable farms: {len(zaid_profitable)} ({zaid_pct:.1f}% of Zaid farms)")

# Crops that beat the Zaid average yield
zaid_mean = zaid["Yield_Tonnes_Ha"].mean()
zaid_crop_yield = zaid.groupby("Crop")["Yield_Tonnes_Ha"].mean().sort_values(ascending=False)
print(f"\nCrops beating Zaid average yield ({zaid_mean:.2f} t/ha):")
print(zaid_crop_yield[zaid_crop_yield > zaid_mean])

# Visualise Z-score distribution by season
SEASON_PALETTE = {"Kharif": "#2196F3", "Rabi": "#4CAF50", "Zaid": "#FF9800"}
fig, ax = plt.subplots(figsize=(12, 4))
for season, color in SEASON_PALETTE.items():
    data = df_anomaly[df_anomaly["Season"] == season]["Yield_Z"]
    ax.hist(data, bins=40, alpha=0.5, color=color, label=season, density=True)
ax.axvline(-3, color="red", linestyle="--", linewidth=1.5, label="|Z|=3 threshold")
ax.axvline( 3, color="red", linestyle="--", linewidth=1.5)
ax.set_xlabel("Yield Z-Score (within-season)")
ax.set_ylabel("Density")
ax.set_title("Yield Z-Score Distribution — Unusual Farms Highlighted", fontweight="bold", fontsize=13)
ax.legend(title="Season")
plt.tight_layout()
plt.show()


---
## 10. Correlation & Relationship Analysis <a id='10-correlation'></a>

In [ ]:
# Correlation matrix of numeric features
numeric_features = [
    'Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day',
    'Soil_pH', 'Soil_Moisture_pct', 'Nitrogen_kg_ha', 'Phosphorus_kg_ha',
    'Potassium_kg_ha', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha',
    'Seed_Quality_Score', 'Yield_Tonnes_Ha', 'Profit_per_Ha',
    'Water_Efficiency_t_per_1000m3', 'Disease_Pest_Risk_pct'
]

corr_matrix = df_clean[numeric_features].corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.4, ax=ax,
            annot_kws={'size': 7}, vmin=-1, vmax=1)
ax.set_title('Correlation Matrix — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Top correlations with Yield
yield_corr = corr_matrix['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#EF5350' if v < 0 else '#42A5F5' for v in yield_corr.values]
ax.barh(yield_corr.index, yield_corr.values, color=colors, edgecolor='white', alpha=0.9)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Yield (Tonnes/Ha)', fontsize=13, fontweight='bold')
ax.set_xlabel('Pearson Correlation Coefficient')
plt.tight_layout()
plt.show()

print('Correlations with Yield_Tonnes_Ha:')
print(yield_corr.round(3))

In [ ]:
# Scatter plots — key relationships coloured by Season
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

scatter_pairs = [
    ('Rainfall_mm',        'Yield_Tonnes_Ha',   'Rainfall vs Yield'),
    ('Fertilizer_kg_ha',   'Yield_Tonnes_Ha',   'Fertilizer vs Yield'),
    ('Seed_Quality_Score', 'Yield_Tonnes_Ha',   'Seed Quality vs Yield'),
]

for ax, (x, y, title) in zip(axes, scatter_pairs):
    for season, color in SEASON_PALETTE.items():
        mask = df_clean['Season'] == season
        ax.scatter(df_clean.loc[mask, x], df_clean.loc[mask, y],
                   alpha=0.3, s=15, color=color, label=season)
    # Regression line (all data)
    x_data = df_clean[x].dropna()
    y_data = df_clean.loc[x_data.index, y]
    m, b = np.polyfit(x_data, y_data, 1)
    xline = np.linspace(x_data.min(), x_data.max(), 100)
    ax.plot(xline, m * xline + b, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel(x)
    ax.set_ylabel('Yield (Tonnes/Ha)')
    ax.set_title(title, fontweight='bold')
    ax.legend(title='Season', fontsize=8)

plt.suptitle('Key Relationships with Crop Yield', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 11. Statistical Testing <a id='11-stats'></a>

We test whether seasonal differences in Yield and Profit are **statistically significant**.

In [ ]:
# --- Kruskal-Wallis test (non-parametric, suitable for non-normal data)
# Tests H0: all seasons have the same distribution

kharif = df_clean[df_clean['Season'] == 'Kharif']
rabi   = df_clean[df_clean['Season'] == 'Rabi']
zaid   = df_clean[df_clean['Season'] == 'Zaid']

for metric in ['Yield_Tonnes_Ha', 'Profit_per_Ha', 'Fertilizer_kg_ha', 'Water_Used_m3']:
    stat, p = stats.kruskal(
        kharif[metric].dropna(),
        rabi[metric].dropna(),
        zaid[metric].dropna()
    )
    sig = '*** SIGNIFICANT' if p < 0.05 else 'not significant'
    print(f'{metric:<35} H={stat:>8.2f}  p={p:.4f}  → {sig}')

In [ ]:
# --- Pairwise Mann-Whitney U tests for Yield between seasons
from itertools import combinations

print('Pairwise Mann-Whitney U Tests — Yield (Tonnes/Ha):\n')
season_pairs = list(combinations(['Kharif', 'Rabi', 'Zaid'], 2))
for s1, s2 in season_pairs:
    g1 = df_clean[df_clean['Season'] == s1]['Yield_Tonnes_Ha'].dropna()
    g2 = df_clean[df_clean['Season'] == s2]['Yield_Tonnes_Ha'].dropna()
    stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
    sig = '*** Significant difference' if p < 0.05 else 'No significant difference'
    print(f'  {s1} vs {s2}: U={stat:.0f}, p={p:.4f} → {sig}')

In [ ]:
# --- Chi-square test: Is Irrigation Method choice independent of Season?
contingency = pd.crosstab(df_clean['Season'], df_clean['Irrigation_Method'])
chi2, p, dof, expected = stats.chi2_contingency(contingency)

print('Chi-Square Test: Irrigation Method vs Season')
print(f'  chi2 = {chi2:.2f}, p = {p:.4f}, dof = {dof}')
if p < 0.05:
    print('  → Irrigation method choice IS significantly associated with Season.')
else:
    print('  → No significant association between irrigation method and season.')

In [ ]:
# --- Point-biserial correlation: Disease risk vs Yield
r, p = stats.pearsonr(df_clean['Disease_Pest_Risk_pct'], df_clean['Yield_Tonnes_Ha'])
print(f'Pearson r (Disease Risk vs Yield): {r:.3f}, p = {p:.4f}')
print('→ Higher disease risk is correlated with', 'lower' if r < 0 else 'higher', 'yield.')

---
## 12. Key Insights & Recommendations <a id='12-insights'></a>

In [ ]:
# Summary dashboard — one chart per key insight
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Avg Yield by Season
ax = axes[0][0]
vals = df_clean.groupby('Season')['Yield_Tonnes_Ha'].mean().reindex(['Kharif', 'Rabi', 'Zaid'])
bars = ax.bar(vals.index, vals.values, color=[SEASON_PALETTE[s] for s in vals.index], alpha=0.9)
ax.set_title('Avg Yield by Season', fontweight='bold')
ax.set_ylabel('Tonnes/Ha')

# 2. Avg Profit by Season
ax = axes[0][1]
vals = df_clean.groupby('Season')['Profit_INR'].mean().reindex(['Kharif', 'Rabi', 'Zaid'])
ax.bar(vals.index, vals.values, color=[SEASON_PALETTE[s] for s in vals.index], alpha=0.9)
ax.axhline(0, color='red', linewidth=1, linestyle='--')
ax.set_title('Avg Profit by Season', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'₹{v/1e5:.1f}L'))

# 3. Disease Risk by Season
ax = axes[0][2]
sns.boxplot(data=df_clean, x='Season', y='Disease_Pest_Risk_pct', hue='Season',
            palette=SEASON_PALETTE, ax=ax, order=['Kharif', 'Rabi', 'Zaid'], legend=False)
ax.set_title('Disease & Pest Risk by Season', fontweight='bold')
ax.set_ylabel('Risk (%)')

# 4. Water Efficiency by Season
ax = axes[1][0]
vals = df_clean.groupby('Season')['Water_Efficiency_t_per_1000m3'].mean().reindex(['Kharif', 'Rabi', 'Zaid'])
ax.bar(vals.index, vals.values, color=[SEASON_PALETTE[s] for s in vals.index], alpha=0.9)
ax.set_title('Avg Water Efficiency by Season', fontweight='bold')
ax.set_ylabel('t / 1000 m³')

# 5. Top crop profit per season (grouped bar)
ax = axes[1][1]
top_crop_profit = df_clean.groupby(['Crop', 'Season'])['Profit_per_Ha'].mean().unstack()
top_crop_profit.plot(kind='bar', ax=ax,
                     color=[SEASON_PALETTE[s] for s in top_crop_profit.columns],
                     edgecolor='white', alpha=0.9, legend=True)
ax.set_title('Profit/Ha by Crop & Season', fontweight='bold')
ax.set_ylabel('INR/Ha')
ax.tick_params(axis='x', rotation=30)
ax.legend(title='Season', fontsize=8)

# 6. Irrigation method share pie chart (Kharif)
ax = axes[1][2]
irr_kharif = df_clean[df_clean['Season'] == 'Kharif']['Irrigation_Method'].value_counts()
ax.pie(irr_kharif.values, labels=irr_kharif.index, autopct='%1.1f%%',
       colors=sns.color_palette('Set2', len(irr_kharif)),
       startangle=90, pctdistance=0.8)
ax.set_title('Irrigation Methods — Kharif', fontweight='bold')

plt.suptitle('Summary Dashboard — Seasonal Agriculture Performance',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### Key Insights

| # | Insight | Evidence |
|---|---------|----------|
| 1 | **Kharif is the highest-yielding and most profitable season** | Highest avg yield and avg profit; highest profitability rate |
| 2 | **Zaid season shows the weakest economic performance** | Negative median profit; lowest profitability rate |
| 3 | **Seasonal differences in yield and profit are statistically significant** | Kruskal-Wallis: Yield H=70.2 (p<0.001), Profit H=97.1 (p<0.001) |
| 4 | **All pairwise season comparisons for yield are significant** | Mann-Whitney U test: Kharif vs Rabi, Kharif vs Zaid, Rabi vs Zaid all p<0.001 |
| 5 | **Kharif receives the most rainfall; Zaid the least** | Environmental analysis confirms seasonal rainfall pattern |
| 6 | **Fertilizer usage does NOT differ significantly across seasons** | Kruskal-Wallis: H=1.06, p=0.589 — farmers apply similar inputs year-round |
| 7 | **Water usage is borderline across seasons** | Kruskal-Wallis: H=5.97, p=0.051 — marginal difference |
| 8 | **Irrigation method choice is NOT significantly tied to season** | Chi-square: chi2=3.51, p=0.742 — usage spread is uniform across seasons |
| 9 | **Disease/Pest risk has negligible effect on yield** | Pearson r=0.014, p=0.364 — other factors dominate yield variation |
| 10 | **Rabi season has moderate but stable returns** | Lower variance in profit compared to Kharif |

---

### Recommendations

1. **Prioritise high-yield crops during Kharif** — the season delivers the best yields and profits; align crop selection (Rice, Sugarcane) accordingly.
2. **Address Zaid season profitability** — negative median profit indicates systemic challenges; targeted subsidies, drought-tolerant crop varieties, or crop diversification are needed.
3. **Right-size fertiliser application** — input usage is uniform across all seasons despite significantly different yield outcomes, suggesting over-application in Zaid and Rabi; precision fertilisation can cut costs without hurting yield.
4. **Invest in Drip irrigation** — it delivers the best water efficiency across all seasons and reduces input costs regardless of season.
5. **Focus disease management on Kharif** — high humidity and temperature create higher risk; early monitoring preserves the season's yield advantage.
6. **Improve seed quality** — correlation analysis shows seed quality as one of the stronger positive predictors of yield; subsidising certified seeds delivers broad impact.
7. **Replicate best-state practices** — state-wise yield heatmap reveals clear high-performers; sharing those agronomic practices with lower-performing states can narrow the gap.

---
*Analysis completed as part of VOIS AICTE Batch1 2026-2027 Major Project.*
---
### Supporting Better Seasonal Agricultural Planning

The findings from this analysis directly enable practical seasonal planning:

| Seasonal Planning Area | Data-Driven Action |
|------------------------|--------------------|
| **Crop calendar** | Prioritise Rice/Sugarcane in Kharif; drought-tolerant crops in Zaid |
| **Input budgeting** | Maintain fertiliser levels in Kharif; reduce in Zaid given low returns |
| **Risk management** | Increase disease surveillance in Kharif (high humidity season) |
| **Water planning** | Adopt Drip irrigation across all seasons for best water efficiency |
| **Market timing** | Use seasonal price variation data to time sales for maximum revenue |

*Analysis completed as part of VOIS AICTE Batch1 2026-2027 Major Project.*
